# Day 2: Pause, Crash, Resume, Retry. Without Doing Anything Twice

**Today's goal:** make the SupportDesk `freeze_card` flow survive a restart, wait for a human, retry a flaky call a limited number of times, and never freeze the same card twice.

## The big analogy: a bank counter with a very good clerk

A customer says: *"Please freeze my card."* Freezing a card is serious, so the clerk follows rules:

| At the bank counter | In our agent | Today's word |
| --- | --- | --- |
| The clerk slips a **bookmark** into the case file and locks it in a drawer. A new clerk can pick up exactly where the last one stopped. | State is saved to a SQLite file after every step | **Checkpoint** |
| Each customer has their own **folder** with a label | `thread_id` = user + `-` + cid (the server builds it) | **Thread** |
| The clerk says *"I need my manager's signature before I go on"* and waits | The graph stops at `interrupt()` until a human answers | **Interrupt / resume** |
| A **tracking page** shows each step as it happens | `stream_mode="updates"` | **Streaming** |
| If the phone line to the card company drops, **redial, but only 3 times** | `RetryPolicy(max_attempts=3)` | **Bounded retry** |
| A **PAID stamp** on the receipt, so a second clerk never charges twice | A key stored in a database table | **Idempotency key** |
| *"Nobody is waiting for a signature on this file. Why are you here?"* | `resume_thread` refuses a thread with no paused work | **Operational boundary** |

**The one idea to remember:** in production, any step can run twice (a retry, a message delivered again, a second click). So we design every step to be **safe to repeat**.

## How to use this notebook
1. Run each cell from top to bottom: click a cell, then press **Shift + Enter**.
2. Read the short story above each cell first. Then run it and compare.
3. Nothing here calls an AI model. It runs offline in a few seconds. The "card system" is a small Python stub.
4. In **Step 7** you will *predict* before you run.
5. The notebook creates a file called `day02_checkpoints.sqlite` next to itself and deletes it at the end.


## Step 0: Setup

We bring in the tools:
- `sqlite3` is Python's built-in small database. One file holds everything.
- `SqliteSaver` is the LangGraph helper that writes the bookmark (checkpoint) into that file.
- `interrupt` and `Command` pause a graph and resume it later.
- `RetryPolicy` says how many times to redial.

We also define:
- `DB_FILE`: the one file that holds **both** the bookmarks and the PAID stamps (the ledger).
- `CARD_SYSTEM`: a pretend card company. Its first call fails with a time-out, to give our retry something to do.
- `reset_world()`: wipes the file and the pretend card company, so every experiment starts clean.

In [1]:
import os
import sqlite3
from typing import TypedDict

from langgraph.checkpoint.memory import InMemorySaver
from langgraph.checkpoint.sqlite import SqliteSaver
from langgraph.graph import END, START, StateGraph
from langgraph.types import Command, RetryPolicy, interrupt

DB_FILE = "day02_checkpoints.sqlite"   # bookmarks AND the ledger live in this one file
CARD_SYSTEM = {"calls": 0, "fail_first": 1}   # the pretend card company: its first call times out


def reset_world(fail_first=1):
    """Start from a clean desk: delete the old file, make a new ledger table, reset the card company."""
    if os.path.exists(DB_FILE):
        os.remove(DB_FILE)
    setup = sqlite3.connect(DB_FILE)
    setup.execute("CREATE TABLE ledger (idempotency_key TEXT PRIMARY KEY, user TEXT)")
    setup.commit()
    setup.close()
    CARD_SYSTEM["calls"] = 0
    CARD_SYSTEM["fail_first"] = fail_first


reset_world()
print("Setup done. Clean desk, empty ledger.")

Setup done. Clean desk, empty ledger.


## Step 1: The case file (the state)

**Analogy.** Every request gets a **case file** with a few labelled boxes. Each step of the graph reads the file and writes into it.

| Box | What goes in it |
| --- | --- |
| `user` | Who is asking (`cust-1`) |
| `question` | What they asked ("Freeze my card") |
| `cid` | A correlation id: one id per conversation |
| `tool` | The tool chosen (`freeze_card`) |
| `approved` | Did the human say yes? |
| `result` | The final sentence for the customer |

These are the same boxes the full SupportDesk uses (you will add more on Days 3 to 5).

In [2]:
class State(TypedDict):   # the labelled boxes on the case file
    user: str
    question: str
    cid: str
    tool: str
    approved: bool
    result: str

print("A case file has", len(State.__annotations__), "boxes:", list(State.__annotations__))

A case file has 6 boxes: ['user', 'question', 'cid', 'tool', 'approved', 'result']


## Step 2: The clerk's four jobs (the nodes)

Each job is one small function. Read them in order.

### Job 1: choose the tool
The agent decides *"this customer needs `freeze_card`"*. (This is the tool that is marked `needs_human=True` in SupportDesk, so a person must say yes.)

### Job 2: ask the manager (the interrupt)
This job does **one thing only**: stop and ask a human.

> **Golden rule.** When a stopped graph resumes, the job **starts again from its first line**. So *nothing with a side effect may sit above `interrupt()`*. If it did, it would run twice. You will see this happen in Experiment 3.

### Job 3: freeze the card (the risky one)
This is the job that touches the outside world. It has three parts:
1. **Call the card company.** The first call times out (our stub). The retry rule redials.
2. **Make the PAID stamp.** The stamp text is `freeze-` + the customer's name. It is the *same every time* for the same customer. Never random, never a timestamp.
3. **Write the stamp into the ledger.** `INSERT OR IGNORE` writes it once. If it is already there, the database says *"0 rows added"* and we skip the freeze.

### Job 4: the fork in the road
After the manager answers, a small function sends the case to **Job 3** if the answer was *yes*, or straight to the **end** if it was *no*.

In [3]:
def choose_tool(state):
    return {"tool": "freeze_card", "result": "freeze requested for " + state["user"]}


def approve_freeze(state):
    # The ONLY line in this job: stop and ask a human. Nothing with side effects above it.
    answer = interrupt({"question": "Freeze the card of " + state["user"] + "?"})
    if answer == "yes":
        return {"approved": True}
    return {"approved": False, "result": "freeze rejected by the human"}


def stamp(key, user):
    """Write the PAID stamp. Returns True the first time, False if the stamp was already there."""
    ledger = sqlite3.connect(DB_FILE)
    with ledger:   # INSERT OR IGNORE is atomic: rowcount 0 means "this key already exists"
        added = ledger.execute("INSERT OR IGNORE INTO ledger VALUES (?, ?)", (key, user)).rowcount
    ledger.close()
    return added == 1


def run_tool(state):
    CARD_SYSTEM["calls"] += 1
    if CARD_SYSTEM["calls"] <= CARD_SYSTEM["fail_first"]:
        raise ConnectionError("card system timed out")   # the retry rule will redial
    key = "freeze-" + state["user"]   # same customer -> same key, even if this job runs twice
    if not stamp(key, state["user"]):
        return {"result": "skipped: " + key + " already done (idempotent)"}
    return {"result": "card for " + state["user"] + " frozen (" + key + ")"}


def after_human(state):
    return "run_tool" if state["approved"] else "end"

print("Four jobs defined.")

Four jobs defined.


## Step 3: Connect the jobs into a graph, and add a redial rule

```text
START -> choose_tool -> approve_freeze --yes--> run_tool -> END
                                       \--no----------------> END
```

Two details matter:

1. **The retry rule is attached to one job only** (`run_tool`): `RetryPolicy(max_attempts=3, retry_on=ConnectionError)`. That means: *redial up to 3 times, wait a little longer each time, and only for a time-out. For any other kind of error, stop at once.* After the third failure the error is raised. (Retrying everything forever is how a small outage becomes a big one.)
2. **The bookmark keeper** (the `saver`) is passed in when we build the graph. That is how every step gets saved.

We wrap this in two helpers:
- `open_app()` makes a **brand-new** connection, saver and graph. That is what a restarted server does, so we call it our *restart*.
- `resume_thread()` is the **operational boundary**: it first asks *"is this file really waiting for a signature?"* and refuses if not.

In [4]:
def build_app(saver):
    graph = StateGraph(State)
    graph.add_node("choose_tool", choose_tool)
    graph.add_node("approve_freeze", approve_freeze)
    graph.add_node("run_tool", run_tool,
                   retry_policy=RetryPolicy(max_attempts=3, initial_interval=0.1, retry_on=ConnectionError))
    graph.add_edge(START, "choose_tool")
    graph.add_edge("choose_tool", "approve_freeze")
    graph.add_conditional_edges("approve_freeze", after_human, {"run_tool": "run_tool", "end": END})
    graph.add_edge("run_tool", END)
    return graph.compile(checkpointer=saver)


def open_app():
    """A brand-new connection, saver and graph: what a restarted server does."""
    connection = sqlite3.connect(DB_FILE, check_same_thread=False)
    return connection, build_app(SqliteSaver(connection))


def resume_thread(app, thread_id, answer):
    """Only resume a thread that is really paused. Never create work by accident."""
    config = {"configurable": {"thread_id": thread_id}}
    if not app.get_state(config).next:   # .next is empty when nothing is waiting
        return "REJECTED: no paused work for thread " + thread_id
    steps = [list(event)[0] for event in app.stream(Command(resume=answer), config, stream_mode="updates")]
    return "steps " + str(steps) + " -> " + app.get_state(config).values["result"]


def waiting_at(app, thread_id):
    return app.get_state({"configurable": {"thread_id": thread_id}}).next

print("Graph builder, restart helper and safe-resume helper are ready.")

Graph builder, restart helper and safe-resume helper are ready.


## Step 4: Two customers ask for a freeze (and the clerk goes home)

Two customers, `cust-1` and `cust-2`, both say *"Freeze my card"*. We run each as its own **thread**.

- The thread id is built by us: `user + "-" + cid`, so `cust-1-c001` and `cust-2-c002`. It is **never** taken from the request. (That rule came from Day 1.)
- We use **streaming**, so we see one update per step instead of waiting for the end. Each of our threads gives two updates: `choose_tool` finished, then `__interrupt__` (*"waiting for a human"*).
- Then we **close the connection**. This is "the server goes down".

Run it. Both customers end up parked at the manager's desk.

In [5]:
connection, app = open_app()
for user, cid in [("cust-1", "c001"), ("cust-2", "c002")]:
    thread = user + "-" + cid
    start = {"user": user, "question": "Freeze my card", "cid": cid, "tool": "", "approved": False, "result": ""}
    events = [list(e)[0] for e in app.stream(start, {"configurable": {"thread_id": thread}}, stream_mode="updates")]
    print("1. " + thread + " streamed updates:", events)
connection.close()   # the "server" goes down. Nothing is in memory any more.
print("   Server is down. Two files are parked in the drawer (the SQLite file).")

1. cust-1-c001 streamed updates: ['choose_tool', '__interrupt__']
1. cust-2-c002 streamed updates: ['choose_tool', '__interrupt__']
   Server is down. Two files are parked in the drawer (the SQLite file).


## Step 5: A new clerk starts work (the restart)

Now `open_app()` builds a **completely new** connection and graph. The new clerk has no memory of what happened. All they have is the drawer.

Watch what they find, and how the safety rules behave:

| Line | What we do | What we expect |
| --- | --- | --- |
| 2 | Ask where `cust-1-c001` is waiting | `('approve_freeze',)`: the bookmark survived the restart |
| 3 | Human says **yes** to `cust-1-c001` | The card is frozen. The first call to the card company timed out, so it was redialled |
| 4 | Count calls to the card company | 2 (one time-out + one retry) |
| 5 | Someone clicks "approve" **again** | `REJECTED`: nothing is paused |
| 6 | Someone sends a thread that never existed | `REJECTED` |
| 7 | Where is `cust-2-c002`? | Still waiting. Resuming one thread did not touch the other |
| 8 | Human says **no** to `cust-2-c002` | `freeze rejected by the human` |

In [6]:
connection, app = open_app()   # the restart: brand-new connection, saver and graph
print("2. After restart, cust-1-c001 waits at:", waiting_at(app, "cust-1-c001"))
print("3. Human approves cust-1-c001:", resume_thread(app, "cust-1-c001", "yes"))
print("4. Card system calls (1 timeout + 1 retry):", CARD_SYSTEM["calls"])
print("5. Second click on the same approval:", resume_thread(app, "cust-1-c001", "yes"))
print("6. Unknown thread:", resume_thread(app, "cust-9-c999", "yes"))
print("7. cust-2-c002 untouched, still waits at:", waiting_at(app, "cust-2-c002"))
print("8. Human rejects cust-2-c002:", resume_thread(app, "cust-2-c002", "no"))
connection.close()

2. After restart, cust-1-c001 waits at: ('approve_freeze',)


3. Human approves cust-1-c001: steps ['approve_freeze', 'run_tool'] -> card for cust-1 frozen (freeze-cust-1)
4. Card system calls (1 timeout + 1 retry): 2
5. Second click on the same approval: REJECTED: no paused work for thread cust-1-c001
6. Unknown thread: REJECTED: no paused work for thread cust-9-c999
7. cust-2-c002 untouched, still waits at: ('approve_freeze',)
8. Human rejects cust-2-c002: steps ['approve_freeze'] -> freeze rejected by the human


## Step 6: The queue delivers the same message again

**Analogy.** A mail room sometimes hands the *same letter* to two clerks. Both read *"freeze cust-1"*.

This is called a **duplicate delivery**, and it is normal in real systems (message queues promise "at least once", not "exactly once").

We simulate it by calling `run_tool` directly for `cust-1` again. The card company is working perfectly this time (`fail_first = 0`).

**Predict:** will the card be frozen a second time? The key is `freeze-cust-1` again, and the stamp is already in the ledger. Run it and see.

In [7]:
CARD_SYSTEM["fail_first"] = 0   # the card company works fine now
duplicate = {"user": "cust-1", "question": "", "cid": "c003", "tool": "freeze_card", "approved": True, "result": ""}
print("9. Duplicate delivery:", run_tool(duplicate)["result"])

ledger = sqlite3.connect(DB_FILE)
print("10. Ledger:", ledger.execute("SELECT * FROM ledger").fetchall())
ledger.close()

9. Duplicate delivery: skipped: freeze-cust-1 already done (idempotent)
10. Ledger: [('freeze-cust-1', 'cust-1')]


One row in the ledger, even though the job ran twice. **That is idempotency:** doing it again gives the same result as doing it once.

### The ten lines at a glance

| # | Line proves | Idea |
| --- | --- | --- |
| 1 | Two updates per thread | Streaming |
| 2 | Bookmark survived the restart | Checkpoint |
| 3 | The job resumed and finished | Interrupt / resume |
| 4 | 2 calls to the card company | Bounded retry |
| 5-6 | Double click and unknown thread rejected | Operational boundary |
| 7 | `cust-2-c002` untouched | Threads are isolated |
| 8 | A "no" ends the flow | The conditional edge |
| 9-10 | Duplicate skipped, one ledger row | Idempotency key |

## Step 7: Your turn. Predict first, then run

**Why predict?** If you guess first and then see the answer, your brain remembers it. If you only run the code, you just watch.

### Experiment 1: the line stays dead

We make the card company fail **5 times** before it would work. Our retry rule only allows **3 attempts**. **What happens?**

<details><summary><b>Click to see the answer</b></summary>

The job **stops with `ConnectionError`** after exactly 3 calls, and the ledger stays **empty**. Retries are *bounded*. An endless retry on a broken line would only make the outage worse.
</details>

In [8]:
reset_world(fail_first=5)   # fresh file, and a card company that fails 5 times in a row
connection, app = open_app()
start = {"user": "cust-1", "question": "Freeze my card", "cid": "c001", "tool": "", "approved": False, "result": ""}
list(app.stream(start, {"configurable": {"thread_id": "cust-1-c001"}}, stream_mode="updates"))   # pause at the human

try:
    resume_thread(app, "cust-1-c001", "yes")
except ConnectionError as error:
    print("The run stopped with:", repr(error))
print("Calls to the card company:", CARD_SYSTEM["calls"], "(max_attempts is 3)")

ledger = sqlite3.connect(DB_FILE)
print("Ledger rows:", ledger.execute("SELECT * FROM ledger").fetchall())
ledger.close()
connection.close()

The run stopped with: ConnectionError('card system timed out')
Calls to the card company: 3 (max_attempts is 3)
Ledger rows: []


### Experiment 2: a PAID stamp that changes every time

What if the key were **not** deterministic? Imagine we put the call counter in it: `freeze-cust-1-1`, then `freeze-cust-1-2`. **How many rows will the ledger hold after two attempts for the same customer?**

<details><summary><b>Click to see the answer</b></summary>

**Two rows.** The second attempt looks like a different request, so the card gets frozen twice. A key must come from *what the request is* (customer + action), never from a random value or the time.
</details>

In [9]:
reset_world()

print("Stable key, two attempts:")
print("  first :", "stamped" if stamp("freeze-cust-1", "cust-1") else "skipped")
print("  second:", "stamped" if stamp("freeze-cust-1", "cust-1") else "skipped")

reset_world()   # empty ledger again, so the two cases do not mix
print("Changing key (call counter inside it), two attempts:")
print("  first :", "stamped" if stamp("freeze-cust-1-1", "cust-1") else "skipped")
print("  second:", "stamped" if stamp("freeze-cust-1-2", "cust-1") else "skipped")

ledger = sqlite3.connect(DB_FILE)
print("Ledger:", ledger.execute("SELECT * FROM ledger").fetchall())
ledger.close()

Stable key, two attempts:
  first : stamped
  second: skipped
Changing key (call counter inside it), two attempts:
  first : stamped
  second: stamped
Ledger: [('freeze-cust-1-1', 'cust-1'), ('freeze-cust-1-2', 'cust-1')]


### Experiment 3: the line above the interrupt

Here is a *bad* version of the manager job. It adds a **side effect above `interrupt()`**: it counts how many times it ran (pretend it sends an SMS). The graph pauses once and is resumed once. **How many times does the SMS line run?**

<details><summary><b>Click to see the answer</b></summary>

**Twice.** On resume the job starts again from its first line, so everything above `interrupt()` runs again. In real life that is two SMS messages (or two charges). Keep the lines above an interrupt free of side effects, as `approve_freeze` does.
</details>

In [10]:
SMS = {"sent": 0}


class Tiny(TypedDict):
    answer: str


def bad_approve(state):
    SMS["sent"] += 1                        # <- a side effect ABOVE the interrupt (pretend: send an SMS)
    answer = interrupt("Freeze the card?")  # pause here
    return {"answer": answer}


tiny = StateGraph(Tiny)
tiny.add_node("bad_approve", bad_approve)
tiny.add_edge(START, "bad_approve")
tiny.add_edge("bad_approve", END)
tiny_app = tiny.compile(checkpointer=InMemorySaver())

config = {"configurable": {"thread_id": "demo"}}
tiny_app.invoke({"answer": ""}, config)                 # runs until the interrupt
print("After pausing, the SMS line ran:", SMS["sent"], "time(s)")
tiny_app.invoke(Command(resume="yes"), config)          # the human answers
print("After resuming, the SMS line ran:", SMS["sent"], "time(s)")

After pausing, the SMS line ran: 1 time(s)
After resuming, the SMS line ran: 2 time(s)


### Experiment 4: two ways to stream

`stream_mode="updates"` gives **only what each step changed**. `stream_mode="values"` gives **the whole case file after each step**. **Which one is better for a progress bar? Which one for a debug log of the full state?**

<details><summary><b>Click to see the answer</b></summary>

`updates` is small and perfect for progress messages ("choose_tool finished"). `values` repeats the entire state each time, which is handy when you want to inspect everything. Run the cell and compare the sizes.
</details>

In [11]:
reset_world()
connection, app = open_app()
start = {"user": "cust-1", "question": "Freeze my card", "cid": "c001", "tool": "", "approved": False, "result": ""}

print("stream_mode='updates'  (only what changed):")
for event in app.stream(start, {"configurable": {"thread_id": "t-updates"}}, stream_mode="updates"):
    print("  ", event)

print()
print("stream_mode='values'  (the whole case file each time):")
for event in app.stream(start, {"configurable": {"thread_id": "t-values"}}, stream_mode="values"):
    print("  ", event)
connection.close()

stream_mode='updates'  (only what changed):
   {'choose_tool': {'tool': 'freeze_card', 'result': 'freeze requested for cust-1'}}
   {'__interrupt__': (Interrupt(value={'question': 'Freeze the card of cust-1?'}, id='016d61c702b039003ddf6718a03a7023', response_schema=None),)}

stream_mode='values'  (the whole case file each time):
   {'user': 'cust-1', 'question': 'Freeze my card', 'cid': 'c001', 'tool': '', 'approved': False, 'result': ''}
   {'user': 'cust-1', 'question': 'Freeze my card', 'cid': 'c001', 'tool': 'freeze_card', 'approved': False, 'result': 'freeze requested for cust-1'}
   {'user': 'cust-1', 'question': 'Freeze my card', 'cid': 'c001', 'tool': 'freeze_card', 'approved': False, 'result': 'freeze requested for cust-1', '__interrupt__': (Interrupt(value={'question': 'Freeze the card of cust-1?'}, id='31a01dd4c6fb25109874be1c0684b70b', response_schema=None),)}


### Tidy up

Delete the demo file so the notebook leaves no mess behind. (All connections are already closed, so Windows will allow it.)

In [12]:
if os.path.exists(DB_FILE):
    os.remove(DB_FILE)
print("Demo file removed:", not os.path.exists(DB_FILE))

Demo file removed: True


## Step 8: Assignment (harden a workflow for interruption and retry)

Harden the SupportDesk `freeze_card` flow. Add a **second interrupt, a supervisor approval**, after the first one (for example when the question mentions a **business account**). Run it in a new notebook cell and show with printed output:

1. A **pause** at the first approval.
2. A **resume in a new connection** (use `open_app()`).
3. A **pause** at the second approval.
4. The **resume** of the second approval.
5. One **injected card-system timeout** that is retried.
6. A **duplicate delivery** that does not freeze the card twice.
7. A **resume of a thread that is not paused**, which is rejected.

Submit the output. Remember the golden rule: nothing with a side effect above an `interrupt()`.

## Production checklist (you will reuse these lines on Day 19)
- [ ] **Durable checkpoints:** the checkpointer is a database or file, and a test resumes a paused thread from a new connection.
- [ ] **Explicit thread_id:** every request has one, and resuming a thread with no pending work is rejected.
- [ ] **Idempotent writes:** `freeze_card` uses a deterministic key stored in a database; a duplicate test shows no second effect.
- [ ] **Bounded retries:** `RetryPolicy` has `max_attempts` and a named error; more failures than attempts ends in a clear error.
- [ ] **No side effects above `interrupt()`** in the same job.

## Recap in one minute
| Idea | The analogy | One-line takeaway |
| --- | --- | --- |
| Checkpoint | Bookmark in a locked drawer | A new process can pick up exactly where the old one stopped. |
| Thread | One labelled folder per customer | Resuming one never touches another. |
| Interrupt | "I need my manager's signature" | Stop and wait, but keep the lines above it free of side effects. |
| Streaming | A tracking page | One small update per step. |
| Bounded retry | Redial 3 times, then stop | Only for known flaky errors, and only a few times. |
| Idempotency key | A PAID stamp | The same request always gives the same stamp, so doing it twice does it once. |
| Operational boundary | "Nobody is waiting here" | Refuse to resume work that is not paused. |

## Tomorrow (Day 3)
Today the state was a handful of boxes. Tomorrow we make state **safe to share**: typed boxes with one owner each, rules for merging updates (reducers), and subgraphs that split SupportDesk into retrieval and tools.
